# PDCP — AI 리스타일 파일럿 (Kaggle 무료 T4x2)

원작 스캔 만화 패널을 **모던 디지털 웹툰 화풍으로 재작화**한다(구도 유지). SDXL + ControlNet(Canny)로 라인을 붙잡아 구성을 보존한다.
PDCP **선택 트랙**(GPU 모델). 기본 트랙(작화 보존, ffmpeg/Claude Code)과 별개.

**사용법**: ① 우측 *Add Data* 로 `ai-restyle/inputs` (job.json 포함)를 Kaggle Dataset 으로 업로드
② Settings → Accelerator = **GPU T4 x2** ③ **Run All** ④ 하단 비교 그리드 확인 → `/kaggle/working/restyled.zip` 다운로드.

In [ ]:
# 1) 의존성 (Kaggle 은 torch 내장)
!pip -q install "diffusers==0.31.0" "transformers>=4.44" accelerate safetensors opencv-python-headless 2>/dev/null
import torch, os, glob, json, math
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) 입력 탐색 — job.json + 패널. Kaggle Dataset 경로 자동 탐색, 없으면 데모 경로.
def find_job():
    for p in glob.glob("/kaggle/input/**/job.json", recursive=True):
        return p
    for p in glob.glob("/kaggle/working/**/job.json", recursive=True):
        return p
    return None
JOB_PATH = find_job()
assert JOB_PATH, "job.json 을 못 찾음 — ai-restyle/inputs 데이터셋을 Add Data 로 붙이세요"
JOB = json.load(open(JOB_PATH))
IN_DIR = os.path.dirname(JOB_PATH) + "/inputs" if os.path.isdir(os.path.dirname(JOB_PATH)+"/inputs") else os.path.dirname(JOB_PATH)
OUT_DIR = "/kaggle/working/restyled"; os.makedirs(OUT_DIR, exist_ok=True)
PRE = JOB["preset"]
print("slug", JOB["slug"], "| engine", JOB["engine"], "| panels", JOB["count"])
print("prompt:", PRE["prompt"])

In [ ]:
# 3) 파이프라인 — SDXL + ControlNet(Canny). T4 16GB 적합: fp16 + model_cpu_offload + vae slicing.
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
controlnet = ControlNetModel.from_pretrained(PRE["controlnet"], torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    PRE["model"], controlnet=controlnet, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.enable_model_cpu_offload()
try: pipe.enable_vae_slicing()
except Exception: pass
print("pipeline ready")

In [ ]:
# 4) 전처리 — Canny 라인(구도 고정) + SDXL 규격(가장 긴 변 1024, 8배수)로 리사이즈.
import cv2, numpy as np
from PIL import Image, ImageDraw, ImageFont

def fit_sdxl(img, target=1024):
    w, h = img.size
    s = target / max(w, h)
    nw, nh = int(round(w*s/8))*8, int(round(h*s/8))*8
    return img.resize((max(nw,512), max(nh,512)), Image.LANCZOS)

def canny(img):
    a = np.array(img.convert("RGB"))
    e = cv2.Canny(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), 90, 200)
    e = np.stack([e]*3, -1)
    return Image.fromarray(e)

def letter_strip(img, texts):
    # 리스타일이 텍스트를 뭉개므로 대사는 클린 폰트로 컷 아래 오버레이(폴백)
    if not texts: return img
    W = img.width; pad=18; fs=max(15, W//34)
    try: font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", fs)
    except Exception: font = ImageFont.load_default()
    lines=[]
    for t in texts:
        words=t.split(); cur=""
        for wd in words:
            if len(cur+" "+wd) > W//(fs//2+1) and cur: lines.append(cur); cur=wd
            else: cur=(cur+" "+wd).strip()
        if cur: lines.append(cur)
        lines.append("")
    if lines and lines[-1]=="": lines.pop()
    ch = pad*2 + len(lines)*(fs+6)
    strip = Image.new("RGB", (W, ch), (251,250,247)); d=ImageDraw.Draw(strip)
    d.rectangle([0,0,7,ch], fill=(46,125,90))
    y=pad
    for ln in lines:
        d.text((20,y), ln, fill=(26,26,26), font=font); y+=fs+6
    out=Image.new("RGB",(W, img.height+ch),(245,244,241)); out.paste(img,(0,0)); out.paste(strip,(0,img.height))
    return out
print("preproc ready")

In [ ]:
# 5) 리스타일 — 패널별 생성. Canny 로 구도 보존 + 프롬프트로 화풍 변경.
g = torch.Generator(device="cuda").manual_seed(7)
results=[]
for p in JOB["panels"]:
    src = Image.open(os.path.join(IN_DIR, p["file"])).convert("RGB")
    base = fit_sdxl(src)
    ctrl = canny(base)
    out = pipe(prompt=PRE["prompt"], negative_prompt=PRE["negative"], image=ctrl,
               controlnet_conditioning_scale=float(PRE["controlScale"]),
               num_inference_steps=int(PRE["steps"]), guidance_scale=float(PRE["guidance"]),
               generator=g).images[0]
    out = out.resize(base.size, Image.LANCZOS)
    final = letter_strip(out, p.get("lettering", []))
    fp = os.path.join(OUT_DIR, p["file"]); final.save(fp, quality=92)
    results.append((p["file"], src, final))
    print("  ✓", p["file"])
print("done", len(results), "panels")

In [ ]:
# 6) 비교 그리드 — 원작 | 리스타일 (제3자 판정용)
from PIL import Image as I
rows=[]
for name, src, res in results:
    H=520
    a=src.convert("RGB"); a=a.resize((int(a.width*H/a.height),H))
    b=res.convert("RGB"); b=b.resize((int(b.width*H/b.height),H))
    row=I.new("RGB",(a.width+b.width+8,H),(200,200,200)); row.paste(a,(0,0)); row.paste(b,(a.width+8,0)); rows.append(row)
W=max(r.width for r in rows); grid=I.new("RGB",(W,sum(r.height for r in rows)+8*len(rows)),(240,240,240))
y=0
for r in rows: grid.paste(r,(0,y)); y+=r.height+8
grid.save("/kaggle/working/compare_grid.jpg", quality=90)
from IPython.display import Image as IPyImage, display
display(IPyImage("/kaggle/working/compare_grid.jpg"))

In [ ]:
# 7) 결과 zip — 로컬로 회수해 ingest.mjs 로 모니터에 적재
import shutil
shutil.make_archive("/kaggle/working/restyled", "zip", OUT_DIR)
print("→ /kaggle/working/restyled.zip 다운로드 후:")
print("   node scripts/comic/pd/ai-restyle/ingest.mjs --workdir work/<slug> --zip restyled.zip")